Visualización CORE 2024

1. Datos de CSV

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from PIL import Image
from google.colab import files
from pathlib import Path

print('Sube el archivo datos_electorales_database_limpia.csv')
subidos = files.upload()
nombre_archivo = 'datos_electorales_database_limpia.csv'
assert nombre_archivo in subidos, 'Debes subir el CSV con ese nombre.'
df = pd.read_csv(nombre_archivo)
print('Filas y columnas:', df.shape)
display(df.head())

2. Comprobar los datos y seleccionar las candidaturas electas

In [ ]:
columnas = ['region', 'electo', 'posicion_lista', 'primero_lista']
assert all(col in df.columns for col in columnas)
assert df[columnas].notna().all().all(), 'Revisar datos faltantes en columnas necesarias.'
assert df['posicion_lista'].ge(1).all()
assert ((df['posicion_lista'] == 1) == (df['primero_lista'] == 'Sí')).all()
assert df['electo'].isin(['Sí', 'No']).all()

electos = df.loc[df['electo'] == 'Sí'].copy()
electos['categoria'] = np.where(electos['posicion_lista'] == 1, 'Primera posición', 'Otras posiciones')
print('Total de candidaturas:', len(df))
print('Candidaturas electas:', len(electos))
print(electos['categoria'].value_counts())
assert len(df) == 2515 and len(electos) == 302
assert (electos['categoria'] == 'Primera posición').sum() == 204

3. Agrupar por región y calcular los porcentajes

In [ ]:
resumen = (electos.groupby(['region', 'categoria']).size()
           .unstack(fill_value=0).reset_index())
for categoria in ['Primera posición', 'Otras posiciones']:
    if categoria not in resumen.columns:
        resumen[categoria] = 0
resumen['total'] = resumen['Primera posición'] + resumen['Otras posiciones']
resumen['porcentaje_primera'] = 100 * resumen['Primera posición'] / resumen['total']

nombres = {'REGIÓN DE ARICA Y PARINACOTA': 'Arica y Parinacota', 'REGIÓN DE TARAPACÁ': 'Tarapacá', 'REGIÓN DE ANTOFAGASTA': 'Antofagasta', 'REGIÓN DE ATACAMA': 'Atacama', 'REGIÓN DE COQUIMBO': 'Coquimbo', 'REGIÓN DE VALPARAÍSO': 'Valparaíso', 'REGIÓN METROPOLITANA DE SANTIAGO': 'Metropolitana', "REGIÓN DEL LIBERTADOR GENERAL BERNARDO O'HIGGINS": "O'Higgins", 'REGIÓN DEL MAULE': 'Maule', 'REGIÓN DE ÑUBLE': 'Ñuble', 'REGIÓN DEL BIOBÍO': 'Biobío', 'REGIÓN DE LA ARAUCANÍA': 'La Araucanía', 'REGIÓN DE LOS RÍOS': 'Los Ríos', 'REGIÓN DE LOS LAGOS': 'Los Lagos', 'REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO': 'Aysén', 'REGIÓN DE MAGALLANES Y DE LA ANTÁRTICA CHILENA': 'Magallanes'}
resumen['region_corta'] = resumen['region'].map(nombres)
assert resumen['region_corta'].notna().all()
resumen = resumen.sort_values('porcentaje_primera', ascending=False)

datos_grafico = resumen.melt(
    id_vars=['region', 'region_corta', 'total', 'porcentaje_primera'],
    value_vars=['Primera posición', 'Otras posiciones'],
    var_name='categoria', value_name='cantidad'
)
datos_grafico['porcentaje'] = 100 * datos_grafico['cantidad'] / datos_grafico['total']
datos_grafico['orden_segmento'] = datos_grafico['categoria'].map({'Primera posición': 0, 'Otras posiciones': 1})
datos_grafico.to_csv('datos_para_visualizacion.csv', index=False)
assert len(resumen) == 16 and len(datos_grafico) == 32
assert np.allclose(datos_grafico.groupby('region_corta')['porcentaje'].sum(), 100)
display(resumen[['region_corta', 'Primera posición', 'Otras posiciones', 'total', 'porcentaje_primera']])

4. Crear el gráfico con Altair

Las barras muestran la composición de los electos en cada región, no representan la probabilidad de elección de cada posición.

In [ ]:
orden_regiones = resumen['region_corta'].tolist()
colores = ['#246A8F', '#E4A76C']

base = alt.Chart(datos_grafico).encode(
    y=alt.Y('region_corta:N', sort=orden_regiones, title=None),
    x=alt.X('sum(porcentaje):Q', stack='zero', scale=alt.Scale(domain=[0, 100]),
            axis=alt.Axis(title='Porcentaje de candidaturas electas', values=[0,20,40,60,80,100], labelExpr="datum.value + ' %'")),
    color=alt.Color('categoria:N',
                    scale=alt.Scale(domain=['Primera posición', 'Otras posiciones'], range=colores),
                    legend=alt.Legend(title='Posición dentro de la lista')),
    order=alt.Order('orden_segmento:Q', sort='ascending'),
    tooltip=[
        alt.Tooltip('region_corta:N', title='Región'),
        alt.Tooltip('categoria:N', title='Posición'),
        alt.Tooltip('cantidad:Q', title='Electos'),
        alt.Tooltip('total:Q', title='Total electos en la región'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje', format='.1f')
    ]
)

barras = base.mark_bar()

grafico = barras.properties(
    width=650,
    height=alt.Step(30),
    title=alt.TitleParams(
        text='¿Qué posición ocupaban las candidaturas electas?',
        subtitle='Elecciones CORE 2024 · Composición porcentual por región'
    )
).configure_view(stroke=None).configure_axis(labelFontSize=12, titleFontSize=12).configure_legend(labelFontSize=12)

grafico

5. Exportar HTML y JPG

El JPG se genera a partir de la imagen exportada por Altair. La carpeta "visualizacion" guardará ambos archivos.

In [ ]:
Path('visualizacion').mkdir(exist_ok=True)
grafico.save('visualizacion/vis_01.html')
grafico.save('visualizacion/vis_01.png', scale_factor=2)
with Image.open('visualizacion/vis_01.png') as img:
    img.convert('RGB').save('visualizacion/vis_01.jpg', quality=95)
print('Archivos creados: visualizacion/vis_01.html y visualizacion/vis_01.jpg')
print('También se creó datos_para_visualizacion.csv')

6. Descargar los resultados para GitHub

In [ ]:
files.download('datos_para_visualizacion.csv')
files.download('visualizacion/vis_01.html')
files.download('visualizacion/vis_01.jpg')